<a href="https://colab.research.google.com/github/ProfeLuisTic1986/Bienvenida-5-B/blob/main/sala_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# redes_ludico.py
import flet as ft
import random
import time

# Uso de la nueva librería `flet-audio` (reemplaza Audio() de ft a partir de Flet 0.26).
# Intentamos importarla; si no está instalada, usamos el control antiguo `ft.Audio`
# como fallback para no romper entornos existentes. Preferimos `ft.Audio` si existe
# porque era el comportamiento previo y funcionaba en el entorno del usuario.
try:
    from flet_audio import Audio as FletAudioControl
except Exception:
    FletAudioControl = None

# Preferir el control nativo `ft.Audio` si está disponible (mantener compatibilidad
# con el comportamiento previo). Si no, usar `flet-audio` si está instalado.
AudioControl = getattr(ft, "Audio", None) or FletAudioControl

# --- Compatibilidad ft.colors / ft.Colors (soporta Flet 0.28 y anteriores) ---
# A veces la API cambia entre versiones: algunas usan `ft.colors` y otras `ft.Colors`.
# Aquí normalizamos ambas referencias y, si ninguna existe, proporcionamos un fallback.
if hasattr(ft, "Colors") and not hasattr(ft, "colors"):
    # En Flet 0.28 existe `ft.Colors` (con C mayúscula). Alias para compatibilidad.
    ft.colors = ft.Colors
elif hasattr(ft, "colors") and not hasattr(ft, "Colors"):
    # En repositorios antiguos puede existir solo `ft.colors` (minúscula). Crear alias.
    ft.Colors = ft.colors
elif not hasattr(ft, "colors") and not hasattr(ft, "Colors"):
    # Ninguna variante existe; definimos un fallback ligero con los colores usados.
    class _ColorsFallback:
        BLUE_700 = "#1976D2"
        BLUE_600 = "#1E88E5"
        BLUE_200 = "#90CAF9"
        BLUE_50  = "#E3F2FD"
        GREEN_600 = "#43A047"
        GREEN_400 = "#66BB6A"
        RED_600 = "#E53935"
        ORANGE_600 = "#FB8C00"
        ORANGE_200 = "#FFCC80"
        ORANGE_50 = "#FFF3E0"
        PURPLE_600 = "#8E24AA"
        PURPLE_50 = "#F3E5F5"
        GREY_700 = "#616161"
        GREY_600 = "#757575"
        GREY_100 = "#F5F5F5"
        YELLOW_50 = "#FFFDE7"
        YELLOW_600 = "#FDD835"
        GOLD = "#FFD700"
        TRANSPARENT = "#00000000"
        WHITE = "#FFFFFF"
        CYAN_50 = "#E0F7FA"
        CYAN_600 = "#00ACC1"
    # Asignar ambas referencias al mismo namespace (la clase)
    ft.Colors = _ColorsFallback
    ft.colors = _ColorsFallback
# --------------------------------------------------------------------

def main(page: ft.Page):
    page.title = "🌐 Aventura en Redes e Internet"
    page.theme_mode = ft.ThemeMode.LIGHT
    page.padding = 20
    page.scroll = "adaptive"

        # --- Audio de fondo ---
    if AudioControl is not None:
        audio_fondo = None
        # Intentar crear el control con la firma más completa y caer a variantes si falla
        for kwargs in (
            {"src": "fondo.mp3", "autoplay": True, "volume": 0.3, "loop": True},
            {"src": "fondo.mp3", "autoplay": True, "volume": 0.3},
            {"src": "fondo.mp3", "volume": 0.3},
        ):
            try:
                audio_fondo = AudioControl(**kwargs)
                break
            except TypeError:
                audio_fondo = None
                continue
            except Exception:
                audio_fondo = None
                break

        if audio_fondo is not None:
            if hasattr(page, "overlay") and page.overlay is not None:
                page.overlay.append(audio_fondo)
            else:
                page.add(audio_fondo)
            # Forzar update para que el audio comience si es necesario
            try:
                page.update()
            except Exception:
                pass
            # Intentar invocar play() si el control lo expone
            try:
                play_fn = getattr(audio_fondo, "play", None)
                if callable(play_fn):
                    play_fn()
            except Exception:
                pass
        else:
            # No se pudo crear el control de audio
            try:
                page.snack_bar = ft.SnackBar(ft.Text("Audio no disponible (control no creado)"))
                page.snack_bar.open = True
                page.update()
            except Exception:
                pass
    else:
        audio_fondo = None


    # Imágenes desde internet
    imagenes = {
        "router": "https://cdn-icons-png.flaticon.com/512/2881/2881341.png",
        "computer": "https://cdn-icons-png.flaticon.com/512/3474/3474360.png",
        "cloud": "https://cdn-icons-png.flaticon.com/512/2920/2920349.png",
        "wifi": "https://cdn-icons-png.flaticon.com/512/2099/2099190.png",
        "firewall": "https://cdn-icons-png.flaticon.com/512/2092/2092665.png",
        "server": "https://cdn-icons-png.flaticon.com/512/1261/1261242.png",
        "network": "https://cdn-icons-png.flaticon.com/512/1183/1183672.png",
        "security": "https://cdn-icons-png.flaticon.com/512/6195/6195699.png",
        "trophy": "https://cdn-icons-png.flaticon.com/512/3176/3176366.png",
        "brain": "https://cdn-icons-png.flaticon.com/512/2784/2784403.png",
        "packet": "https://cdn-icons-png.flaticon.com/512/2920/2920231.png",
    }

    # Preguntas de trivia CON PISTAS Y EXPLICACIONES
    preguntas_trivia = [
        {
            "pregunta": "¿Qué significa HTTP?",
            "pista": "💡 Pista: Es el protocolo que usas cada vez que visitas una página web. La 'H' significa HyperText.",
            "opciones": ["HyperText Transfer Protocol", "High Transfer Text Protocol", "Home Tool Transfer Protocol"],
            "correcta": 0,
            "explicacion": "HTTP (HyperText Transfer Protocol) es el protocolo que permite la transferencia de páginas web entre servidores y navegadores.",
            "imagen": "https://cdn-icons-png.flaticon.com/512/2165/2165004.png"
        },
        {
            "pregunta": "¿Cuál es la dirección IP de localhost?",
            "pista": "💡 Pista: Es una dirección especial que siempre apunta a tu propia computadora. Empieza con 127...",
            "opciones": ["192.168.1.1", "127.0.0.1", "10.0.0.1"],
            "correcta": 1,
            "explicacion": "127.0.0.1 es la dirección IP de localhost, que siempre se refiere a tu propia máquina. Es como decir 'yo mismo' en términos de red.",
            "imagen": "https://cdn-icons-png.flaticon.com/512/1183/1183621.png"
        },
        {
            "pregunta": "¿Qué capa del modelo OSI maneja el enrutamiento?",
            "pista": "💡 Pista: El modelo OSI tiene 7 capas. Esta capa se encarga de encontrar el mejor camino para los datos. Piensa en 'red'...",
            "opciones": ["Capa de Aplicación", "Capa de Red", "Capa Física"],
            "correcta": 1,
            "explicacion": "La Capa de Red (Layer 3) es responsable del enrutamiento de paquetes entre diferentes redes usando direcciones IP.",
            "imagen": "https://cdn-icons-png.flaticon.com/512/2920/2920277.png"
        },
        {
            "pregunta": "¿Qué protocolo se usa para enviar correos?",
            "pista": "💡 Pista: Sus siglas significan 'Simple Mail Transfer Protocol'. La palabra clave es 'Mail'.",
            "opciones": ["FTP", "SMTP", "DNS"],
            "correcta": 1,
            "explicacion": "SMTP (Simple Mail Transfer Protocol) es el protocolo estándar para enviar correos electrónicos a través de Internet.",
            "imagen": "https://cdn-icons-png.flaticon.com/512/561/561127.png"
        },
        {
            "pregunta": "¿Qué significa DNS?",
            "pista": "💡 Pista: Es como la 'agenda telefónica' de Internet. Convierte nombres (google.com) en números (IPs).",
            "opciones": ["Domain Name System", "Digital Network Service", "Data Network Security"],
            "correcta": 0,
            "explicacion": "DNS (Domain Name System) traduce nombres de dominio legibles (como google.com) a direcciones IP que las computadoras entienden.",
            "imagen": "https://cdn-icons-png.flaticon.com/512/2920/2920229.png"
        },
        {
            "pregunta": "¿Qué puerto usa HTTPS por defecto?",
            "pista": "💡 Pista: HTTP usa el puerto 80. HTTPS (la versión segura) usa un número mayor... 443.",
            "opciones": ["80", "443", "8080"],
            "correcta": 1,
            "explicacion": "HTTPS usa el puerto 443 por defecto. Es la versión segura y encriptada de HTTP.",
            "imagen": "https://cdn-icons-png.flaticon.com/512/2092/2092573.png"
        },
        {
            "pregunta": "¿Qué dispositivo conecta múltiples redes?",
            "pista": "💡 Pista: Este dispositivo 'enruta' el tráfico entre diferentes redes. Lo tienes en casa para conectarte a Internet.",
            "opciones": ["Switch", "Router", "Hub"],
            "correcta": 1,
            "explicacion": "Un Router conecta múltiples redes y dirige el tráfico entre ellas, como el router de tu casa que conecta tu red local a Internet.",
            "imagen": imagenes["router"]
        },
        {
            "pregunta": "¿Qué significa WWW?",
            "pista": "💡 Pista: Son las tres letras que ves al inicio de muchas páginas web. Significa 'World Wide...'",
            "opciones": ["World Wide Web", "World Web Wide", "Wide World Web"],
            "correcta": 0,
            "explicacion": "WWW (World Wide Web) es el sistema de documentos interconectados accesibles a través de Internet, inventado por Tim Berners-Lee en 1989.",
            "imagen": "https://cdn-icons-png.flaticon.com/512/1183/1183672.png"
        }
    ]

    # Estado del juego
    puntuacion = {"valor": 0}
    pregunta_actual = {"indice": 0}
    mostrar_pista_activa = {"valor": False}

    contenido = ft.Column(scroll="adaptive", expand=True)

    # Controles expuestos para Makey Makey / teclado externo
    simulator_controls = {"enviar": None, "toggle": None, "reiniciar": None}

    def mostrar_pista_global():
        """Muestra la pista de la pregunta actual en un SnackBar (útil para input externo)."""
        try:
            idx = pregunta_actual["indice"]
            if 0 <= idx < len(preguntas_trivia):
                p = preguntas_trivia[idx]
                page.snack_bar = ft.SnackBar(ft.Text(p.get("pista", "")))
                page.snack_bar.open = True
                page.update()
        except Exception:
            pass

    def mostrar_inicio():
        contenido.controls.clear()
        contenido.controls.append(
            ft.Container(
                content=ft.Column([
                    ft.Image(
                        src=imagenes["network"],
                        width=150,
                        height=150,
                        fit=ft.ImageFit.CONTAIN,
                    ),
                    ft.Text("🌐 Aventura en Redes e Internet", size=32, weight=ft.FontWeight.BOLD, color=ft.colors.BLUE_700),
                    ft.Text("Aprende sobre redes de forma divertida", size=18, color=ft.colors.GREY_700),
                    ft.Divider(height=30, color=ft.colors.TRANSPARENT),

                    ft.Container(
                        content=ft.Row([
                            ft.Image(src="https://cdn-icons-png.flaticon.com/512/2991/2991148.png", width=40, height=40),
                            ft.Text("Jugar Trivia", size=18, color=ft.colors.WHITE),
                        ], alignment=ft.MainAxisAlignment.CENTER),
                        on_click=lambda _: mostrar_trivia(),
                        bgcolor=ft.colors.BLUE_600,
                        padding=20,
                        border_radius=10,
                        width=300,
                        ink=True,
                    ),

                    ft.Container(height=15),

                    ft.Container(
                        content=ft.Row([
                            ft.Image(src=imagenes["router"], width=40, height=40),
                            ft.Text("Simulador de Red", size=18, color=ft.colors.WHITE),
                        ], alignment=ft.MainAxisAlignment.CENTER),
                        on_click=lambda _: mostrar_simulador(),
                        bgcolor=ft.colors.GREEN_600,
                        padding=20,
                        border_radius=10,
                        width=300,
                        ink=True,
                    ),

                    ft.Container(height=15),

                    ft.Container(
                        content=ft.Row([
                            ft.Image(src=imagenes["brain"], width=40, height=40),
                            ft.Text("Aprende Conceptos", size=18, color=ft.colors.WHITE),
                        ], alignment=ft.MainAxisAlignment.CENTER),
                        on_click=lambda _: mostrar_conceptos(),
                        bgcolor=ft.colors.PURPLE_600,
                        padding=20,
                        border_radius=10,
                        width=300,
                        ink=True,
                    ),

                    ft.Container(height=15),

                    ft.Container(
                        content=ft.Row([
                            ft.Image(src="https://cdn-icons-png.flaticon.com/512/3524/3524388.png", width=40, height=40),
                            ft.Text("Visualizar Red", size=18, color=ft.colors.WHITE),
                        ], alignment=ft.MainAxisAlignment.CENTER),
                        on_click=lambda _: mostrar_visualizacion(),
                        bgcolor=ft.colors.ORANGE_600,
                        padding=20,
                        border_radius=10,
                        width=300,
                        ink=True,
                    ),
                    ft.Divider(height=20, color=ft.colors.TRANSPARENT),

                    # Leyenda de teclas para Makey Makey
                    ft.Container(
                        content=ft.Column([
                            ft.Text("Controles Makey Makey (teclas):", size=14, weight=ft.FontWeight.W_600),
                            ft.Row([
                                ft.Container(content=ft.Text("↑: Trivia"), padding=6, bgcolor=ft.colors.BLUE_50, border_radius=6, margin=ft.margin.only(right=8)),
                                ft.Container(content=ft.Text("←: Simulador"), padding=6, bgcolor=ft.colors.GREEN_400 if hasattr(ft.colors, 'GREEN_400') else ft.colors.GREEN_600, border_radius=6, margin=ft.margin.only(right=8)),
                                ft.Container(content=ft.Text("↓: Conceptos"), padding=6, bgcolor=ft.colors.PURPLE_50, border_radius=6, margin=ft.margin.only(right=8)),
                                ft.Container(content=ft.Text("→: Visualizar"), padding=6, bgcolor=ft.colors.ORANGE_50, border_radius=6),
                            ], alignment=ft.MainAxisAlignment.CENTER),
                            ft.Row([
                                ft.Container(content=ft.Text("Space / Click: Acción / Seleccionar"), padding=6, bgcolor=ft.colors.YELLOW_50, border_radius=6, margin=ft.margin.only(right=8)),
                                ft.Container(content=ft.Text("Front pins: ↑,↓,←,→,Space,Click"), padding=6, bgcolor=ft.colors.BLUE_50, border_radius=6, margin=ft.margin.only(right=8)),
                                ft.Container(content=ft.Text("Back pins (W,A,S,D,F,G): acciones adicionales"), padding=6, bgcolor=ft.colors.GREY_100, border_radius=6),
                            ], alignment=ft.MainAxisAlignment.CENTER),
                            ft.Row([
                                ft.Container(content=ft.Text("H: Mostrar Pista"), padding=6, bgcolor=ft.colors.YELLOW_50, border_radius=6, margin=ft.margin.only(right=8)),
                                ft.Container(content=ft.Text("N: Siguiente Pregunta"), padding=6, bgcolor=ft.colors.BLUE_50, border_radius=6, margin=ft.margin.only(right=8)),
                                ft.Container(content=ft.Text("Volver al menú: Space / Enter / Esc / B"), padding=6, bgcolor=ft.colors.GREY_100, border_radius=6),
                            ], alignment=ft.MainAxisAlignment.CENTER),
                        ], horizontal_alignment=ft.CrossAxisAlignment.CENTER),
                        padding=12,
                        bgcolor=ft.colors.GREY_100,
                        border_radius=8,
                        margin=ft.margin.only(top=6),
                    ),
                ], horizontal_alignment=ft.CrossAxisAlignment.CENTER),
                padding=40,
            )
        )
        page.update()

    def mostrar_trivia():
        puntuacion["valor"] = 0
        pregunta_actual["indice"] = 0
        mostrar_pista_activa["valor"] = False
        random.shuffle(preguntas_trivia)
        mostrar_pregunta()

    def mostrar_pregunta():
        if pregunta_actual["indice"] >= len(preguntas_trivia):
            mostrar_resultado_final()
            return

        pregunta = preguntas_trivia[pregunta_actual["indice"]]
        mostrar_pista_activa["valor"] = False
        contenido.controls.clear()

        # Contenedor para la pista (inicialmente oculto)
        pista_container = ft.Container(
            content=ft.Column([
                ft.Row([
                    ft.Icon(ft.Icons.LIGHTBULB, color=ft.colors.YELLOW_600, size=30),
                    ft.Text(pregunta["pista"], size=14, color=ft.colors.GREY_700, italic=True),
                ]),
            ]),
            padding=15,
            bgcolor=ft.colors.YELLOW_50,
            border_radius=10,
            border=ft.border.all(2, ft.colors.YELLOW_600),
            visible=False,
        )

        def toggle_pista(_):
            mostrar_pista_activa["valor"] = not mostrar_pista_activa["valor"]
            pista_container.visible = mostrar_pista_activa["valor"]
            boton_pista.text = "🙈 Ocultar Pista" if mostrar_pista_activa["valor"] else "💡 Ver Pista"
            page.update()

        boton_pista = ft.ElevatedButton(
            "💡 Ver Pista",
            on_click=toggle_pista,
            style=ft.ButtonStyle(
                bgcolor=ft.colors.YELLOW_600,
                color=ft.colors.WHITE,
            ),
        )

        def verificar_respuesta(indice):
            if indice == pregunta["correcta"]:
                puntuacion["valor"] += 10
                mostrar_feedback(True, pregunta["explicacion"])
            else:
                mostrar_feedback(False, pregunta["explicacion"])

        botones_opciones = []
        for i, opcion in enumerate(pregunta["opciones"]):
            botones_opciones.append(
                ft.Container(
                    content=ft.Text(opcion, size=16, text_align=ft.TextAlign.CENTER),
                    on_click=lambda _, idx=i: verificar_respuesta(idx),
                    bgcolor=ft.colors.BLUE_50,
                    padding=20,
                    border_radius=10,
                    width=400,
                    border=ft.border.all(2, ft.colors.BLUE_200),
                    ink=True,
                    margin=ft.margin.only(bottom=10),
                )
            )

        contenido.controls.append(
            ft.Container(
                content=ft.Column([
                    ft.Row([
                        ft.IconButton(ft.Icons.ARROW_BACK, on_click=lambda _: mostrar_inicio()),
                        ft.Text(f"Puntuación: {puntuacion['valor']}", size=20, weight=ft.FontWeight.BOLD),
                    ], alignment=ft.MainAxisAlignment.SPACE_BETWEEN),

                    ft.Text(f"Pregunta {pregunta_actual['indice'] + 1} de {len(preguntas_trivia)}",
                            size=16, color=ft.colors.GREY_600),
                    ft.Divider(height=20, color=ft.colors.TRANSPARENT),

                    ft.Image(
                        src=pregunta["imagen"],
                        width=120,
                        height=120,
                        fit=ft.ImageFit.CONTAIN,
                    ),

                    ft.Container(
                        content=ft.Text(pregunta["pregunta"], size=24, weight=ft.FontWeight.BOLD, text_align=ft.TextAlign.CENTER),
                        padding=20,
                        bgcolor=ft.colors.BLUE_50,
                        border_radius=10,
                    ),

                    ft.Divider(height=15, color=ft.colors.TRANSPARENT),

                    boton_pista,
                    ft.Divider(height=8, color=ft.colors.TRANSPARENT),
                    # Leyenda rápida de teclas para la trivia
                    ft.Row([
                        ft.Container(content=ft.Text("H = Mostrar/Ocultar Pista"), padding=6, bgcolor=ft.colors.YELLOW_50, border_radius=6, margin=ft.margin.only(right=8)),
                        ft.Container(content=ft.Text("N = Siguiente Pregunta"), padding=6, bgcolor=ft.colors.BLUE_50, border_radius=6),
                        ft.Container(content=ft.Text("Volver al menú: Space / Enter / Esc / B"), padding=6, bgcolor=ft.colors.GREY_100, border_radius=6, margin=ft.margin.only(left=8)),
                    ], alignment=ft.MainAxisAlignment.CENTER),

                    ft.Container(height=10),

                    pista_container,

                    ft.Divider(height=20, color=ft.colors.TRANSPARENT),

                    ft.Column(botones_opciones, horizontal_alignment=ft.CrossAxisAlignment.CENTER),
                ], horizontal_alignment=ft.CrossAxisAlignment.CENTER),
                padding=20,
            )
        )
        page.update()

    def mostrar_feedback(correcto, explicacion):
        contenido.controls.clear()

        if correcto:
            icono_url = "https://cdn-icons-png.flaticon.com/512/5610/5610944.png"
            mensaje = "¡Correcto! 🎉"
            color = ft.colors.GREEN_600
        else:
            icono_url = "https://cdn-icons-png.flaticon.com/512/753/753345.png"
            mensaje = "Incorrecto 😔"
            color = ft.colors.RED_600

        contenido.controls.append(
            ft.Container(
                content=ft.Column([
                    ft.Image(src=icono_url, width=120, height=120),
                    ft.Text(mensaje, size=32, weight=ft.FontWeight.BOLD, color=color),

                    ft.Divider(height=20, color=ft.colors.TRANSPARENT),

                    ft.Container(
                        content=ft.Column([
                            ft.Row([
                                ft.Icon(ft.Icons.SCHOOL, color=ft.colors.CYAN_600, size=30),
                                ft.Text("Explicación:", size=18, weight=ft.FontWeight.BOLD, color=ft.colors.CYAN_600),
                            ]),
                            ft.Text(explicacion, size=15, color=ft.colors.GREY_700),
                        ]),
                        padding=20,
                        bgcolor=ft.colors.CYAN_50,
                        border_radius=10,
                        border=ft.border.all(2, ft.colors.CYAN_600),
                        width=500,
                    ),

                    ft.Divider(height=30, color=ft.colors.TRANSPARENT),

                    ft.ElevatedButton(
                        "Siguiente Pregunta ➡️",
                        on_click=lambda _: siguiente_pregunta(),
                        style=ft.ButtonStyle(padding=20, bgcolor=ft.colors.BLUE_600, color=ft.colors.WHITE),
                        width=250,
                    ),
                ], horizontal_alignment=ft.CrossAxisAlignment.CENTER),
                padding=40,
                alignment=ft.alignment.center,
            )
        )
        page.update()

    def siguiente_pregunta():
        pregunta_actual["indice"] += 1
        mostrar_pregunta()

    def mostrar_resultado_final():
        contenido.controls.clear()
        porcentaje = (puntuacion["valor"] / (len(preguntas_trivia) * 10)) * 100

        if porcentaje >= 80:
            mensaje = "¡Excelente! Eres un experto en redes 🏆"
            color = ft.colors.GOLD
            imagen_resultado = imagenes["trophy"]
        elif porcentaje >= 60:
            mensaje = "¡Bien hecho! Vas por buen camino 👍"
            color = ft.colors.GREEN_600
            imagen_resultado = "https://cdn-icons-png.flaticon.com/512/2767/2767146.png"
        else:
            mensaje = "Sigue practicando, ¡tú puedes! 💪"
            color = ft.colors.BLUE_600
            imagen_resultado = "https://cdn-icons-png.flaticon.com/512/2936/2936719.png"

        contenido.controls.append(
            ft.Container(
                content=ft.Column([
                    ft.Image(src=imagen_resultado, width=150, height=150),
                    ft.Text("🎉 Juego Terminado", size=32, weight=ft.FontWeight.BOLD),
                    ft.Divider(height=20, color=ft.colors.TRANSPARENT),
                    ft.Text(f"Puntuación Final: {puntuacion['valor']}/{len(preguntas_trivia) * 10}",
                            size=28, color=color, weight=ft.FontWeight.BOLD),
                    ft.Text(f"Porcentaje: {int(porcentaje)}%", size=20, color=color),
                    ft.Text(mensaje, size=20, color=color),
                    ft.Divider(height=30, color=ft.colors.TRANSPARENT),
                    ft.Row([
                        ft.ElevatedButton("Jugar de Nuevo", on_click=lambda _: mostrar_trivia(),
                                          style=ft.ButtonStyle(padding=15)),
                        ft.ElevatedButton("Menú Principal", on_click=lambda _: mostrar_inicio(),
                                          style=ft.ButtonStyle(padding=15)),
                    ], alignment=ft.MainAxisAlignment.CENTER),
                ], horizontal_alignment=ft.CrossAxisAlignment.CENTER),
                padding=40,
            )
        )
        page.update()

    def mostrar_simulador():
        contenido.controls.clear()

        estado_red = {
            "conectado": False,
            "paquetes": 0,
            "velocidad": "0 Mbps",
            "latencia": "0 ms",
            "paquetes_perdidos": 0,
            "animando": False
        }

        # Contenedores animados para los paquetes
        paquete1 = ft.Container(
            content=ft.Image(src=imagenes["packet"], width=30, height=30),
            visible=False,
            left=50,
            top=100,
        )
        paquete2 = ft.Container(
            content=ft.Image(src=imagenes["packet"], width=30, height=30),
            visible=False,
            left=50,
            top=100,
        )

        def enviar_paquete(_):
            if not estado_red["conectado"]:
                return

            estado_red["paquetes"] += 1
            estado_red["velocidad"] = f"{random.randint(50, 100)} Mbps"
            estado_red["latencia"] = f"{random.randint(10, 50)} ms"

            # Simular pérdida de paquetes ocasional
            if random.random() < 0.1:  # 10% de probabilidad
                estado_red["paquetes_perdidos"] += 1

            actualizar_simulador()

        def toggle_conexion(_):
            estado_red["conectado"] = not estado_red["conectado"]
            if not estado_red["conectado"]:
                estado_red["velocidad"] = "0 Mbps"
                estado_red["latencia"] = "0 ms"
            actualizar_simulador()

        def reiniciar_stats(_):
            estado_red["paquetes"] = 0
            estado_red["paquetes_perdidos"] = 0
            estado_red["velocidad"] = "0 Mbps" if not estado_red["conectado"] else f"{random.randint(50, 100)} Mbps"
            estado_red["latencia"] = "0 ms" if not estado_red["conectado"] else f"{random.randint(10, 50)} ms"
            actualizar_simulador()

        # Exponer controles para que el manejador de teclado los pueda invocar
        simulator_controls["enviar"] = enviar_paquete
        simulator_controls["toggle"] = toggle_conexion
        simulator_controls["reiniciar"] = reiniciar_stats

        def actualizar_simulador():
            contenido.controls.clear()

            color_conexion = ft.colors.GREEN_600 if estado_red["conectado"] else ft.colors.RED_600
            texto_estado = "Conectado ✓" if estado_red["conectado"] else "Desconectado ✗"

            # Calcular tasa de éxito
            tasa_exito = 100
            if estado_red["paquetes"] > 0:
                tasa_exito = ((estado_red["paquetes"] - estado_red["paquetes_perdidos"]) / estado_red["paquetes"]) * 100

            contenido.controls.append(
                ft.Container(
                    content=ft.Column([
                        ft.Row([
                            ft.IconButton(ft.Icons.ARROW_BACK, on_click=lambda _: mostrar_inicio()),
                            ft.Text("🔌 Simulador de Red Interactivo", size=28, weight=ft.FontWeight.BOLD),
                        ]),

                        ft.Divider(height=20, color=ft.colors.TRANSPARENT),

                        # Visualización de la red
                        ft.Container(
                            content=ft.Column([
                                ft.Row([
                                    ft.Container(
                                        content=ft.Column([
                                            ft.Image(src=imagenes["computer"], width=80, height=80),
                                            ft.Text("Tu PC", size=14, weight=ft.FontWeight.BOLD),
                                        ], horizontal_alignment=ft.CrossAxisAlignment.CENTER),
                                        padding=20,
                                        bgcolor=ft.colors.BLUE_50,
                                        border_radius=10,
                                        border=ft.border.all(2, ft.colors.BLUE_200),
                                    ),

                                    ft.Container(
                                        content=ft.Text("──▶", size=30, color=color_conexion, weight=ft.FontWeight.BOLD),
                                        margin=ft.margin.symmetric(horizontal=20),
                                    ),

                                    ft.Container(
                                        content=ft.Column([
                                            ft.Image(src=imagenes["router"], width=80, height=80),
                                            ft.Text("Router", size=14, weight=ft.FontWeight.BOLD),
                                        ], horizontal_alignment=ft.CrossAxisAlignment.CENTER),
                                        padding=20,
                                        bgcolor=ft.colors.ORANGE_50,
                                        border_radius=10,
                                        border=ft.border.all(2, ft.colors.ORANGE_200),
                                    ),

                                    ft.Container(
                                        content=ft.Text("──▶", size=30, color=color_conexion, weight=ft.FontWeight.BOLD),
                                        margin=ft.margin.symmetric(horizontal=20),
                                    ),

                                    ft.Container(
                                        content=ft.Column([
                                            ft.Image(src=imagenes["cloud"], width=80, height=80),
                                            ft.Text("Internet", size=14, weight=ft.FontWeight.BOLD),
                                        ], horizontal_alignment=ft.CrossAxisAlignment.CENTER),
                                        padding=20,
                                        bgcolor=ft.colors.PURPLE_50,
                                        border_radius=10,
                                        border=ft.border.all(2, ft.colors.PURPLE_600),
                                    ),
                                ], alignment=ft.MainAxisAlignment.CENTER),
                            ]),
                            padding=30,
                        ),

                        ft.Divider(height=20, color=ft.colors.TRANSPARENT),

                        # Panel de estadísticas
                        ft.Container(
                            content=ft.Column([
                                ft.Text("📊 Estadísticas de Red", size=20, weight=ft.FontWeight.BOLD),
                                ft.Divider(height=10, color=ft.colors.TRANSPARENT),
                                ft.Row([
                                    ft.Column([
                                        ft.Text(f"Estado: {texto_estado}", size=16, weight=ft.FontWeight.BOLD, color=color_conexion),
                                        ft.Text(f"📦 Paquetes enviados: {estado_red['paquetes']}", size=14),
                                        ft.Text(f"❌ Paquetes perdidos: {estado_red['paquetes_perdidos']}", size=14, color=ft.colors.RED_600),
                                    ], spacing=5),
                                    ft.VerticalDivider(width=20),
                                    ft.Column([
                                        ft.Text(f"⚡ Velocidad: {estado_red['velocidad']}", size=14),
                                        ft.Text(f"⏱️ Latencia: {estado_red['latencia']}", size=14),
                                        ft.Text(f"✅ Tasa de éxito: {tasa_exito:.1f}%", size=14, color=ft.colors.GREEN_600),
                                    ], spacing=5),
                                ], alignment=ft.MainAxisAlignment.CENTER),
                            ]),
                            padding=20,
                            bgcolor=ft.colors.GREY_100,
                            border_radius=10,
                            alignment=ft.alignment.center,
                        ),

                        ft.Divider(height=20, color=ft.colors.TRANSPARENT),

                        # Leyenda de teclas para el simulador
                        ft.Container(
                            content=ft.Row([
                                ft.Container(content=ft.Text("E: Enviar Paquete"), padding=6, bgcolor=ft.colors.BLUE_50, border_radius=6, margin=ft.margin.only(right=8)),
                                ft.Container(content=ft.Text("C: Conectar/Desconectar"), padding=6, bgcolor=ft.colors.GREEN_400 if hasattr(ft.colors, 'GREEN_400') else ft.colors.GREEN_600, border_radius=6, margin=ft.margin.only(right=8)),
                                ft.Container(content=ft.Text("R: Reiniciar Stats"), padding=6, bgcolor=ft.colors.ORANGE_50, border_radius=6),
                                ft.Container(content=ft.Text("Volver al menú: Space / Enter / Esc / B"), padding=6, bgcolor=ft.colors.GREY_100, border_radius=6, margin=ft.margin.only(left=8)),
                            ], alignment=ft.MainAxisAlignment.CENTER),
                            padding=8,
                            bgcolor=ft.colors.GREY_100,
                            border_radius=6,
                            margin=ft.margin.only(bottom=8),
                        ),

                        # Botones de control
                        ft.Row([
                            ft.ElevatedButton(
                                "🔌 Conectar" if not estado_red["conectado"] else "🔴 Desconectar",
                                on_click=toggle_conexion,
                                style=ft.ButtonStyle(
                                    bgcolor=ft.colors.GREEN_600 if not estado_red["conectado"] else ft.colors.RED_600,
                                    color=ft.colors.WHITE,
                                    padding=20,
                                ),
                                width=200,
                            ),
                            ft.ElevatedButton(
                                "📤 Enviar Paquete",
                                on_click=enviar_paquete,
                                style=ft.ButtonStyle(
                                    bgcolor=ft.colors.BLUE_600,
                                    color=ft.colors.WHITE,
                                    padding=20,
                                ),
                                width=200,
                                disabled=not estado_red["conectado"],
                            ),
                            ft.ElevatedButton(
                                "🔄 Reiniciar Stats",
                                on_click=reiniciar_stats,
                                style=ft.ButtonStyle(
                                    bgcolor=ft.colors.ORANGE_600,
                                    color=ft.colors.WHITE,
                                    padding=20,
                                ),
                                width=200,
                            ),
                        ], alignment=ft.MainAxisAlignment.CENTER, wrap=True),

                        ft.Divider(height=20, color=ft.colors.TRANSPARENT),

                        # Información educativa
                        ft.Container(
                            content=ft.Column([
                                ft.Row([
                                    ft.Icon(ft.Icons.INFO, color=ft.colors.BLUE_600, size=30),
                                    ft.Text("¿Qué está pasando?", size=16, weight=ft.FontWeight.BOLD, color=ft.colors.BLUE_600),
                                ]),
                                ft.Text(
                                    "• Los paquetes son pequeñas unidades de datos que viajan por la red\n"
                                    "• La latencia es el tiempo que tarda un paquete en llegar a su destino\n"
                                    "• La velocidad indica cuántos datos pueden transferirse por segundo\n"
                                    "• Los paquetes pueden perderse por congestión o errores en la red",
                                    size=13,
                                    color=ft.colors.GREY_700,
                                ),
                            ]),
                            padding=20,
                            bgcolor=ft.colors.BLUE_50,
                            border_radius=10,
                            border=ft.border.all(2, ft.colors.BLUE_200),
                            width=700,
                        ),
                    ], horizontal_alignment=ft.CrossAxisAlignment.CENTER),
                    padding=20,
                )
            )
            page.update()

        actualizar_simulador()

    # Manejador de teclado (Makey Makey emula teclas). Mapear teclas a acciones.
    def on_keyboard_event(e: ft.KeyboardEvent):
        # Obtener llave en formato simple
        key = str(e.key) if e.key is not None else ""
        k = key.replace(" ", "").lower()

        try:
            # Back-header keys (WASD) - compatibility with Makey Makey back pins
            if k in ("w", "arrowup", "up"):
                mostrar_trivia()
            elif k in ("a", "arrowleft", "left"):
                mostrar_simulador()
            elif k in ("s", "arrowdown", "down"):
                mostrar_conceptos()
            elif k in ("d", "arrowright", "right"):
                mostrar_visualizacion()
            elif k in (" ", "space", "spacebar", "enter", "return", "esc", "escape", "b"):
                mostrar_inicio()
            elif k in ("n", "arrowright", "arrowright"):
                # siguiente pregunta en la trivia
                try:
                    siguiente_pregunta()
                except Exception:
                    pass
            elif k in ("h",):
                mostrar_pista_global()
            elif k in ("e",):
                if simulator_controls.get("enviar"):
                    simulator_controls["enviar"](None)
            elif k in ("c",):
                if simulator_controls.get("toggle"):
                    simulator_controls["toggle"](None)
            elif k in ("r",):
                if simulator_controls.get("reiniciar"):
                    simulator_controls["reiniciar"](None)
            elif k in ("b",):
                mostrar_inicio()
        finally:
            # actualizar la página por si alguna acción lo requiere
            try:
                page.update()
            except Exception:
                pass

    page.on_keyboard_event = on_keyboard_event

    def mostrar_conceptos():
        contenido.controls.clear()

        conceptos = [
            {
                "titulo": "IP Address",
                "desc": "Identificador único de cada dispositivo en la red, como una dirección postal digital.",
                "imagen": "https://cdn-icons-png.flaticon.com/512/1183/1183621.png"
            },
            {
                "titulo": "Firewall",
                "desc": "Barrera de seguridad que protege tu red de accesos no autorizados.",
                "imagen": imagenes["firewall"]
            },
            {
                "titulo": "Router",
                "desc": "Dispositivo que dirige el tráfico de datos entre tu red local e Internet.",
                "imagen": imagenes["router"]
            },
            {
                "titulo": "HTTPS",
                "desc": "Protocolo seguro para transferir datos en la web, protege tu información.",
                "imagen": imagenes["security"]
            },
            {
                "titulo": "DNS",
                "desc": "Sistema que traduce nombres de dominio (google.com) a direcciones IP.",
                "imagen": "https://cdn-icons-png.flaticon.com/512/2920/2920229.png"
            },
            {
                "titulo": "Wi-Fi",
                "desc": "Tecnología que permite conexión inalámbrica a redes de Internet.",
                "imagen": imagenes["wifi"]
            },
            {
                "titulo": "Servidor",
                "desc": "Computadora que proporciona servicios y recursos a otros dispositivos en la red.",
                "imagen": imagenes["server"]
            },
            {
                "titulo": "Protocolo TCP/IP",
                "desc": "Conjunto de reglas que permiten la comunicación entre dispositivos en Internet.",
                "imagen": "https://cdn-icons-png.flaticon.com/512/2920/2920277.png"
            },
        ]

        tarjetas = []
        for concepto in conceptos:
            tarjetas.append(
                ft.Container(
                    content=ft.Row([
                        ft.Image(src=concepto["imagen"], width=60, height=60, fit=ft.ImageFit.CONTAIN),
                        ft.Container(width=20),
                        ft.Column([
                            ft.Text(concepto["titulo"], size=20, weight=ft.FontWeight.BOLD),
                            ft.Text(concepto["desc"], size=14, color=ft.colors.GREY_700),
                        ], expand=True),
                    ]),
                    padding=20,
                    bgcolor=ft.colors.BLUE_50,
                    border_radius=10,
                    margin=ft.margin.only(bottom=15),
                    width=700,
                    border=ft.border.all(2, ft.colors.BLUE_200),
                )
            )

        contenido.controls.append(
            ft.Container(
                content=ft.Column([
                    ft.Row([
                        ft.IconButton(ft.Icons.ARROW_BACK, on_click=lambda _: mostrar_inicio()),
                        ft.Text("📚 Conceptos de Redes", size=28, weight=ft.FontWeight.BOLD),
                    ]),

                    ft.Divider(height=20, color=ft.colors.TRANSPARENT),

                    ft.Column(tarjetas, scroll="adaptive"),
                ], horizontal_alignment=ft.CrossAxisAlignment.CENTER),
                padding=20,
            )
        )
        page.update()

    def mostrar_visualizacion():
        contenido.controls.clear()

        contenido.controls.append(
            ft.Container(
                content=ft.Column([
                    ft.Row([
                        ft.IconButton(ft.Icons.ARROW_BACK, on_click=lambda _: mostrar_inicio()),
                        ft.Text("🌐 Visualización de Red", size=28, weight=ft.FontWeight.BOLD),
                    ]),

                    ft.Divider(height=20, color=ft.colors.TRANSPARENT),

                    ft.Text("Arquitectura de una Red Típica", size=22, weight=ft.FontWeight.BOLD, color=ft.colors.BLUE_700),

                    ft.Container(height=20),

                    ft.Container(
                        content=ft.Row([
                            ft.Image(src=imagenes["cloud"], width=100, height=100),
                            ft.Text("INTERNET", size=24, weight=ft.FontWeight.BOLD),
                        ], alignment=ft.MainAxisAlignment.CENTER),
                        bgcolor=ft.colors.PURPLE_50,
                        padding=20,
                        border_radius=10,
                        border=ft.border.all(3, ft.colors.PURPLE_600),
                    ),

                    ft.Text("↓", size=40, color=ft.colors.GREY_700),

                    ft.Container(
                        content=ft.Row([
                            ft.Image(src=imagenes["firewall"], width=80, height=80),
                            ft.Column([
                                ft.Text("FIREWALL", size=20, weight=ft.FontWeight.BOLD),
                                ft.Text("Protección y seguridad", size=12, color=ft.colors.GREY_700),
                            ]),
                        ], alignment=ft.MainAxisAlignment.CENTER),
                        bgcolor="#FFCDD2",
                        padding=15,
                        border_radius=10,
                        border=ft.border.all(3, ft.colors.RED_600),
                    ),

                    ft.Text("↓", size=40, color=ft.colors.GREY_700),

                    ft.Container(
                        content=ft.Row([
                            ft.Image(src=imagenes["router"], width=80, height=80),
                            ft.Column([
                                ft.Text("ROUTER", size=20, weight=ft.FontWeight.BOLD),
                                ft.Text("Distribuye la conexión", size=12, color=ft.colors.GREY_700),
                            ]),
                        ], alignment=ft.MainAxisAlignment.CENTER),
                        bgcolor=ft.colors.ORANGE_50,
                        padding=15,
                        border_radius=10,
                        border=ft.border.all(3, ft.colors.ORANGE_600),
                    ),

                    ft.Text("↓", size=40, color=ft.colors.GREY_700),

                    ft.Row([
                        ft.Container(
                            content=ft.Column([
                                ft.Image(src=imagenes["computer"], width=60, height=60),
                                ft.Text("PC", size=14, weight=ft.FontWeight.BOLD),
                            ], horizontal_alignment=ft.CrossAxisAlignment.CENTER),
                            bgcolor=ft.colors.BLUE_50,
                            padding=15,
                            border_radius=10,
                            border=ft.border.all(2, ft.colors.BLUE_600),
                        ),
                        ft.Container(
                            content=ft.Column([
                                ft.Image(src="https://cdn-icons-png.flaticon.com/512/3474/3474360.png", width=60, height=60),
                                ft.Text("Laptop", size=14, weight=ft.FontWeight.BOLD),
                            ], horizontal_alignment=ft.CrossAxisAlignment.CENTER),
                            bgcolor=ft.colors.BLUE_50,
                            padding=15,
                            border_radius=10,
                            border=ft.border.all(2, ft.colors.BLUE_600),
                        ),
                        ft.Container(
                            content=ft.Column([
                                ft.Image(src="https://cdn-icons-png.flaticon.com/512/3474/3474361.png", width=60, height=60),
                                ft.Text("Móvil", size=14, weight=ft.FontWeight.BOLD),
                            ], horizontal_alignment=ft.CrossAxisAlignment.CENTER),
                            bgcolor=ft.colors.BLUE_50,
                            padding=15,
                            border_radius=10,
                            border=ft.border.all(2, ft.colors.BLUE_600),
                        ),
                    ], alignment=ft.MainAxisAlignment.CENTER),

                    ft.Divider(height=30, color=ft.colors.TRANSPARENT),

                    ft.Container(
                        content=ft.Row([
                            ft.Image(src="https://cdn-icons-png.flaticon.com/512/2810/2810051.png", width=40, height=40),
                            ft.Text(
                                "Así es como los dispositivos se conectan a Internet de forma segura",
                                size=14,
                                color=ft.colors.GREY_700,
                            ),
                        ], alignment=ft.MainAxisAlignment.CENTER),
                        padding=20,
                        bgcolor=ft.colors.YELLOW_50,
                        border_radius=10,
                    ),
                ], horizontal_alignment=ft.CrossAxisAlignment.CENTER),
                padding=20,
            )
        )
        page.update()

    # Iniciar la aplicación
    page.add(contenido)
    mostrar_inicio()

if __name__ == "__main__":
    ft.app(target=main)